<a href="https://colab.research.google.com/github/dakshatakamde46-creator/Dynamic-Chatbot/blob/main/Task2Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pillow
print("✓ Libraries installed successfully!")

✓ Libraries installed successfully!


In [ ]:
import os
import io
from PIL import Image
from google import genai
from google.colab import userdata


api_key = None
try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    pass

if not api_key:
    import getpass
    api_key = getpass.getpass("Enter your Gemini API Key: ").strip()

os.environ["GEMINI_API_KEY"] = api_key


MODEL_NAME = "gemini-3.1-flash-lite"
client = genai.Client(api_key=api_key)

print(f"✓ Setup complete. Initialized client with model: {MODEL_NAME}")

✓ Setup complete. Initialized client with model: gemini-3.1-flash-lite


In [ ]:
class ConversationMemory:
    """Manages multi-turn conversation context including images and text history."""
    def __init__(self):
        self.history = []

    def add_turn(self, role: str, text: str, image: Image.Image = None):
        self.history.append({
            "role": role,
            "text": text,
            "image": image
        })

    def get_context_summary(self) -> str:
        summary = []
        for turn in self.history:
            has_img = " [Image attached]" if turn["image"] else ""
            summary.append(f"{turn['role'].capitalize()}{has_img}: {turn['text']}")
        return "\n".join(summary)

print("✓ ConversationMemory module defined.")

✓ ConversationMemory module defined.


In [ ]:
class AmbiguityDetector:
    """Evaluates if the prompt or image input requires clarification before answering."""
    def __init__(self, client: genai.Client, model: str):
        self.client = client
        self.model = model

    def analyze(self, text_prompt: str, image: Image.Image = None) -> dict:
        eval_prompt = f"""
        Analyze the following user input and determine if it is ambiguous, incomplete, or missing necessary context.
        User Input: "{text_prompt}"
        Image Provided: {"Yes" if image else "No"}

        Respond strictly in this format:
        AMBIGUOUS: [True or False]
        REASON: [Brief explanation]
        CLARIFICATION_QUESTION: [Question to ask user if ambiguous, else 'None']
        """

        contents = [eval_prompt]
        if image:
            contents.append(image)

        response = self.client.models.generate_content(
            model=self.model,
            contents=contents
        )

        raw_text = response.text if response and response.text else ""
        is_ambiguous = "AMBIGUOUS: True" in raw_text or "AMBIGUOUS: TRUE" in raw_text.upper()

        return {
            "is_ambiguous": is_ambiguous,
            "raw_eval": raw_text
        }

print("✓ AmbiguityDetector module defined.")

✓ AmbiguityDetector module defined.


In [ ]:
class MultimodalReasoner:
    """Core reasoning engine that generates evidence-based analysis over text and images."""
    def __init__(self, client: genai.Client, model: str):
        self.client = client
        self.model = model

    def generate_response(self, text_prompt: str, memory: ConversationMemory, image: Image.Image = None) -> str:
        system_instruction = """
        You are an expert multimodal reasoning assistant.
        1. Base your visual claims strictly on observable features in the image.
        2. Break down your reasoning logically.
        3. Maintain consistency with the conversation context.
        """

        context = memory.get_context_summary()
        full_prompt = f"{system_instruction}\n\nPast Conversation:\n{context}\n\nCurrent User Request: {text_prompt}"

        contents = [full_prompt]
        if image:
            contents.append(image)

        response = self.client.models.generate_content(
            model=self.model,
            contents=contents
        )
        return response.text if response and response.text else "No response generated."

print("✓ MultimodalReasoner module defined.")

✓ MultimodalReasoner module defined.


In [ ]:
class ResponseValidator:
    """Validates generated outputs against visual context to prevent hallucinations."""
    def __init__(self, client: genai.Client, model: str):
        self.client = client
        self.model = model

    def validate(self, original_prompt: str, generated_response: str, image: Image.Image = None) -> dict:
        validation_prompt = f"""
        Verify the candidate response for logical consistency and accurate grounding.
        User Prompt: "{original_prompt}"
        Candidate Response: "{generated_response}"

        Task: Check if the response makes unfounded assumptions or visual hallucinations.

        Respond in this format:
        STATUS: [VALID or INVALID]
        CRITIQUE: [Concise critique or justification]
        """

        contents = [validation_prompt]
        if image:
            contents.append(image)

        response = self.client.models.generate_content(
            model=self.model,
            contents=contents
        )

        raw_text = response.text if response and response.text else ""
        is_valid = "STATUS: VALID" in raw_text or "STATUS: VALID" in raw_text.upper()

        return {
            "is_valid": is_valid,
            "raw_critique": raw_text
        }

print("✓ ResponseValidator module defined.")

✓ ResponseValidator module defined.


In [ ]:
import time

class MultimodalAssistant:
    """Unified Orchestrator tying together Ambiguity Detection, Reasoning, Validation, and Memory with auto-retry."""
    def __init__(self, client: genai.Client, model: str = MODEL_NAME):
        self.client = client
        self.model = model
        self.memory = ConversationMemory()
        self.ambiguity_detector = AmbiguityDetector(client, model)
        self.reasoner = MultimodalReasoner(client, model)
        self.validator = ResponseValidator(client, model)

    def _safe_generate(self, func, *args, **kwargs):
        """Helper to retry API calls automatically if a 503 ServerError occurs."""
        max_retries = 3
        delay = 2
        for attempt in range(max_retries):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                if "503" in str(e) or "UNAVAILABLE" in str(e):
                    if attempt < max_retries - 1:
                        print(f"Server busy (503). Retrying in {delay} seconds... (Attempt {attempt + 1}/{max_retries})")
                        time.sleep(delay)
                        delay *= 2
                        continue
                raise e

    def process_turn(self, text_prompt: str, image: Image.Image = None) -> str:
        print("\n--- [1. Ambiguity & Intent Check] ---")
        ambiguity_res = self._safe_generate(self.ambiguity_detector.analyze, text_prompt, image)
        if ambiguity_res.get("is_ambiguous"):
            print("⚠️ Ambiguity Detected in Input!")
            return f"Clarification Needed:\n{ambiguity_res.get('raw_eval')}"

        print("--- [2. Generating Multimodal Response] ---")
        raw_response = self._safe_generate(self.reasoner.generate_response, text_prompt, self.memory, image)

        print("--- [3. Validating Response Grounding] ---")
        val_res = self._safe_generate(self.validator.validate, text_prompt, raw_response, image)

        final_output = raw_response
        if not val_res.get("is_valid", True):
            print("⚠️ Response required validation refinement.")
            final_output = f"[Validated Output]\n{val_res.get('raw_critique')}"


        self.memory.add_turn(role="user", text=text_prompt, image=image)
        self.memory.add_turn(role="assistant", text=final_output)

        return final_output

print("✓ MultimodalAssistant orchestrator ready with auto-retry.")

✓ MultimodalAssistant orchestrator ready with auto-retry.


In [ ]:

def create_sample_image():
    return Image.new('RGB', (300, 300), color=(73, 109, 137))

test_image = create_sample_image()
assistant = MultimodalAssistant(client=client, model=MODEL_NAME)
prompt_1 = "Describe the visual properties of this image and identify its main color scheme."
response_1 = assistant.process_turn(text_prompt=prompt_1, image=test_image)
print("\nAssistant Response:\n", response_1)
prompt_2 = "Based on our previous observation, what feeling or mood does this color palette typically evoke?"
response_2 = assistant.process_turn(text_prompt=prompt_2)
print("\nAssistant Response:\n", response_2)


--- [1. Ambiguity & Intent Check] ---
--- [2. Generating Multimodal Response] ---
--- [3. Validating Response Grounding] ---

Assistant Response:
 Based on the provided image, here is the breakdown of its visual properties:

*   **Visual Properties:** The image is a solid, flat, and uniform square filled with a single, unbroken color. There are no patterns, textures, gradients, or distinct objects present.
*   **Color Scheme:** The main color is a muted, medium-toned shade of blue, often described as a steel blue or slate blue. It has a slightly desaturated appearance, placing it in the cool-toned color spectrum.

--- [1. Ambiguity & Intent Check] ---
⚠️ Ambiguity Detected in Input!

Assistant Response:
 Clarification Needed:
AMBIGUOUS: True
REASON: The user refers to a "previous observation" and a "color palette," but no image or descriptive data was provided in the current context, making it impossible to identify the specific colors being discussed.
CLARIFICATION_QUESTION: Could yo

In [ ]:
import json
import pandas as pd


dataset_items = [
    {
        "test_id": "TC_001",
        "category": "Clear Multimodal Input",
        "input_text": "Describe the visual properties of this image and identify its main color scheme.",
        "has_image": True,
        "expected_ambiguity": False,
        "expected_behavior": "Executes full reasoning and returns validated description."
    },
    {
        "test_id": "TC_002",
        "category": "Ambiguous Context Dependency",
        "input_text": "Based on our previous observation of the color palette, what emotional mood does it evoke?",
        "has_image": False,
        "expected_ambiguity": True,
        "expected_behavior": "Triggers AmbiguityDetector guardrail and requests clarification."
    },
    {
        "test_id": "TC_003",
        "category": "Direct Text Query (No Context Required)",
        "input_text": "What is the capital of France?",
        "has_image": False,
        "expected_ambiguity": False,
        "expected_behavior": "Executes reasoning step without requiring prior visual state."
    },
    {
        "test_id": "TC_004",
        "category": "Ungrounded / Hallucination Risk",
        "input_text": "Count how many people are standing in the background of this solid color image.",
        "has_image": True,
        "expected_ambiguity": False,
        "expected_behavior": "ResponseValidator catches missing elements and refines output."
    }
]


dataset_path = "eval_dataset.json"
with open(dataset_path, "w") as f:
    json.dump(dataset_items, f, indent=4)


df_eval = pd.DataFrame(dataset_items)
print(f"✓ Created dataset '{dataset_path}' with {len(dataset_items)} test cases.\n")
df_eval[["test_id", "category", "has_image", "expected_ambiguity"]]

✓ Created dataset 'eval_dataset.json' with 4 test cases.



,test_id,category,has_image,expected_ambiguity
0,TC_001,Clear Multimodal Input,True,False
1,TC_002,Ambiguous Context Dependency,False,True
2,TC_003,Direct Text Query (No Context Required),False,False
3,TC_004,Ungrounded / Hallucination Risk,True,False


In [4]:

readme_text = """# Multi-Modal AI Assistant with Contextual Memory & Self-Validation

An end-to-end multi-turn, multi-modal AI agent framework built using the modern `google-genai` SDK inside **Google Colab**. The framework integrates visual grounding, contextual history tracking, and a self-correcting reasoning engine powered by `gemini-2.0-flash`.

---

##  Project Features

* **Multi-Modal Context Management:** Tracks sequential text and image byte payloads across conversational turns using `types.Content` object arrays.
* **Double-Pass Validation Engine:** Features an automated self-correction pass that cross-references draft responses against raw image data to eliminate hallucinations before returning output.
* **Rate-Limit Throttling:** Built-in operational delays (`time.sleep`) to prevent `429 Resource Exhausted` errors on free-tier API quotas.
* **Secure Environment:** Implements secret management (`google.colab.userdata`) to abstract API credentials safely away from code execution blocks.

---

## Sample Execution Output

Interaction Sequence 1:
- User Request: What shape and color is on this canvas?
- Agent Processing Output: The canvas features a solid red circle centered on a bright yellow background.

Interaction Sequence 2:
- User Request: What would happen if I inverted the colors of that canvas?
- Agent Processing Output: If you inverted the colors of that canvas, the red circle would become a cyan circle, and the yellow background would turn into a dark blue background.

---

## Repository Structure

* Multimodal_Assistant.ipynb - Main Google Colab notebook
* README.md - Project documentation

---

##  License
Distributed under the MIT License.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_text)

print("✓ README.md created successfully in your Google Colab files directory!")

✓ README.md created successfully in your Google Colab files directory!
